In [ ]:
!pip install -q plotly pandas

In [ ]:
from google.colab import files
files.upload()   # select static.csv and dynamic.csv

Saving dynamic.csv to dynamic.csv
Saving static.csv to static.csv


{'dynamic.csv': b'time,cpu_util,mem_util,resp_time,throughput,sla_violations,migrations,saturated_hosts,active_hosts\r\n0.1,0.2000,0.1300,0.504,6.000,0,0,0,6\r\n5.1,0.1996,0.1299,0.503,5.989,0,4,0,6\r\n10.1,0.1993,0.1299,0.503,5.980,0,10,0,6\r\n15.1,0.1991,0.1299,0.503,5.972,0,10,0,6\r\n20.1,0.1988,0.1298,0.503,5.965,0,10,0,6\r\n25.1,0.1987,0.1298,0.503,5.961,0,10,0,6\r\n30.1,0.1986,0.1298,0.503,5.959,0,10,0,6\r\n35.1,0.1987,0.1298,0.503,5.960,0,10,0,6\r\n40.1,0.1988,0.1298,0.503,5.963,0,12,0,6\r\n45.1,0.1989,0.1298,0.504,5.968,0,20,0,6\r\n50.1,0.1992,0.1299,0.505,5.976,0,22,0,6\r\n55.1,0.1995,0.1299,0.505,5.985,0,22,0,6\r\n60.1,0.1998,0.1300,0.506,5.995,0,22,0,6\r\n65.1,0.2002,0.1300,0.506,6.005,0,22,0,6\r\n70.1,0.2005,0.1301,0.506,6.015,0,22,0,6\r\n75.1,0.2008,0.1301,0.507,6.024,0,24,0,6\r\n80.1,0.2010,0.1302,0.507,6.031,0,27,0,6\r\n85.1,0.2012,0.1302,0.507,6.037,0,29,0,6\r\n90.1,0.2013,0.1302,0.507,6.040,0,29,0,6\r\n95.1,0.2014,0.1302,0.506,6.041,0,29,0,6\r\n100.1,0.2013,0.1302,0.50

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

s = pd.read_csv("static.csv")     # or "results/static.csv" locally
d = pd.read_csv("dynamic.csv")

# ---- KPI summary table ----
def kpis(df):
    return {
        "Avg CPU util (%)":      round(df.cpu_util.mean() * 100, 1),
        "Avg memory util (%)":   round(df.mem_util.mean() * 100, 1),
        "Avg response time (s)": round(df.resp_time[df.resp_time > 0].mean(), 2),
        "Avg throughput (req/s)":round(df.throughput.mean(), 2),
        "SLA violations (total)":int(df.sla_violations.sum()),
        "VM migrations":         int(df.migrations.max()),
        "Avg active hosts":      round(df.active_hosts.mean(), 2),
    }
summary = pd.DataFrame({"Static": kpis(s), "Dynamic": kpis(d)})
display(summary)

# ---- charts ----
panels = [
    ("cpu_util",        "CPU utilization (%)",     100),
    ("mem_util",        "Memory utilization (%)",  100),
    ("resp_time",       "Response time (s)",       1),
    ("throughput",      "Throughput (req/s)",      1),
    ("sla_violations",  "SLA violations per window", 1),
    ("migrations",      "Cumulative VM migrations", 1),
    ("saturated_hosts", "Saturated hosts",         1),
    ("active_hosts",    "Active hosts",            1),
]
fig = make_subplots(rows=4, cols=2, subplot_titles=[p[1] for p in panels],
                    vertical_spacing=0.08)
for i, (col, title, k) in enumerate(panels):
    r, c = i // 2 + 1, i % 2 + 1
    show = (i == 0)
    fig.add_trace(go.Scatter(x=s.time, y=s[col]*k, name="Static",
                  line=dict(color="#1f77b4"), showlegend=show), row=r, col=c)
    fig.add_trace(go.Scatter(x=d.time, y=d[col]*k, name="Dynamic",
                  line=dict(color="#ff7f0e"), showlegend=show), row=r, col=c)
    fig.update_xaxes(title_text="time (s)", row=r, col=c)
fig.update_layout(height=1100, title="Static vs Dynamic Resource Allocation")
fig.show()

,Static,Dynamic
Avg CPU util (%),50.30,49.30
Avg memory util (%),17.50,17.70
Avg response time (s),1.38,3.16
Avg throughput (req/s),15.08,15.02
SLA violations (total),158.00,229.00
VM migrations,0.00,92.00
Avg active hosts,6.00,5.95
